In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName('MostObscureSuperheroes').getOrCreate()

schema = StructType([
    StructField('id', IntegerType()),
    StructField('name', StringType())
])

names = spark.read.schema(schema).option('sep', ' ').csv('./MarvelNames.txt')

lines = spark.read.text('MarvelGraph.txt')

connections = lines.withColumn('id', func.split(func.col('value'), ' ')[0]) \
    .withColumn('connections', func.size(func.split(func.col('value'), ' ')) - 1) \
    .groupBy('id').agg(func.sum('connections').alias('connections'))

# Find the minimum number of connections
minConnectionCount = connections.agg(func.min('connections')).first()[0]

# Filter down to only heroes matching that minimum (handles ties)
minConnections = connections.filter(func.col('connections') == minConnectionCount)

# Join with names to get readable output
minConnectionsWithNames = minConnections.join(names, 'id')

print('The following heroes are the most obscure, with ' + str(minConnectionCount) + ' co-appearance(s):')
minConnectionsWithNames.select('name').show(minConnectionsWithNames.count(), False)

spark.stop()